In [1]:
import statistics
import time

from datasets import load_dataset
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

/Users/soundwave77/projects/distillation_course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Выбор модели и контекста применения
- **Модель:** `Harsha901/tinybert-imdb-sentiment-analysis-model`.
- **Задача:** определить, отзыв на фильм положительный или отрицательный.
- **Сценарий:** оффлайн-инференс на фиксированном наборе данных.
- **Нагрузка для оценки:** 500 примеров из IMDb; считаются latency, throughput и accuracy.
- **Платформа:** Apple Silicon GPU через PyTorch MPS backend. Тестирую модель на личном ноутбук локально.

In [2]:
if not torch.backends.mps.is_available():
    raise RuntimeError('MPS недоступен: этот ноутбук рассчитан на запуск с device=mps.')

cfg = {
    'model_name': 'Harsha901/tinybert-imdb-sentiment-analysis-model',
    'device': 'mps',
    'batch_size': 16,
    'max_length': 256,
    'warmup_steps': 10,
    'profile_steps': 20,
    'imdb_samples': 500,
    'shuffle_seed': 42,
}


In [3]:
tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
base_model = AutoModelForSequenceClassification.from_pretrained(cfg['model_name']).to(cfg['device']).eval()

dataset = load_dataset('imdb', split='test').shuffle(seed=cfg['shuffle_seed']).select(range(cfg['imdb_samples']))
texts = list(dataset['text'])
labels = torch.tensor(list(dataset['label']), dtype=torch.long)

enc = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=cfg['max_length'],
    return_tensors='pt',
)

features = {k: v for k, v in enc.items()}
print('Loaded IMDb samples:', len(texts))
print('Tokenized shape:', {k: tuple(v.shape) for k, v in features.items()})


Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 73/73 [00:00<00:00, 13654.92it/s]

Loaded IMDb samples: 500
Tokenized shape: {'input_ids': (500, 256), 'token_type_ids': (500, 256), 'attention_mask': (500, 256)}


## 2) Метрики производительности
Метрики считаются на 500 примерах IMDb.

In [4]:
def iter_batches(features, labels, batch_size, device):
    n = labels.shape[0]
    for i in range(0, n, batch_size):
        sl = slice(i, min(i + batch_size, n))
        batch = {k: v[sl].to(device) for k, v in features.items()}
        y = labels[sl].to(device)
        yield batch, y

def run_inference(model, features, labels, device, batch_size):
    latencies_ms = []
    all_preds = []
    all_logits = []

    with torch.inference_mode():
        for batch, y in iter_batches(features, labels, batch_size, device):
            t0 = time.perf_counter()
            out = model(**batch)
            torch.mps.synchronize()
            t1 = time.perf_counter()

            latencies_ms.append((t1 - t0) * 1000.0)
            logits = out.logits.detach().float().cpu()
            preds = logits.argmax(dim=-1)
            all_logits.append(logits)
            all_preds.append(preds)

    preds = torch.cat(all_preds, dim=0)
    logits = torch.cat(all_logits, dim=0)
    labels_cpu = labels.cpu()
    accuracy = (preds == labels_cpu).float().mean().item()

    mean_ms = statistics.fmean(latencies_ms)
    p50_ms = statistics.median(latencies_ms)
    p95_ms = sorted(latencies_ms)[int(0.95 * (len(latencies_ms) - 1))]
    total_samples = labels.shape[0]
    total_time_s = sum(latencies_ms) / 1000.0
    throughput = total_samples / total_time_s

    return {
        'mean_batch_latency_ms': mean_ms,
        'p50_batch_latency_ms': p50_ms,
        'p95_batch_latency_ms': p95_ms,
        'throughput_samples_s': throughput,
        'accuracy': accuracy,
        'preds': preds,
        'logits': logits,
    }

def benchmark_model(model, features, labels, cfg):
    model.eval()

    with torch.inference_mode():
        warm_batches = 0
        for batch, _ in iter_batches(features, labels, cfg['batch_size'], cfg['device']):
            _ = model(**batch)
            warm_batches += 1
            if warm_batches >= cfg['warmup_steps']:
                break
        torch.mps.synchronize()

    res = run_inference(model, features, labels, cfg['device'], cfg['batch_size'])

    current_mem_mb = torch.mps.current_allocated_memory() / (1024 ** 2)

    res.update({
        'params_million': sum(p.numel() for p in model.parameters()) / 1e6,
        'model_size_mb': sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2),
        'current_memory_mb': current_mem_mb,
    })
    return res


In [5]:
baseline = benchmark_model(base_model, features, labels, cfg)
baseline_view = {k: v for k, v in baseline.items() if k not in {'preds', 'logits'}}
baseline_view

{'mean_batch_latency_ms': 21.29630865613308,
 'p50_batch_latency_ms': 21.263354499751586,
 'p95_batch_latency_ms': 21.724166999774752,
 'throughput_samples_s': 733.6952263555868,
 'accuracy': 0.8859999775886536,
 'params_million': 14.350874,
 'model_size_mb': 54.744239807128906,
 'current_memory_mb': 55.121337890625}

## 3) Профилирование bottlenecks

In [6]:
def profile_bottlenecks(model, features, labels, cfg, top_k=20):
    activities = [torch.profiler.ProfilerActivity.CPU]

    with torch.inference_mode():
        with torch.profiler.profile(
            activities=activities,
            record_shapes=True,
            profile_memory=True,
            with_flops=False,
            acc_events=True,
        ) as prof:
            steps = 0
            for batch, _ in iter_batches(features, labels, cfg['batch_size'], cfg['device']):
                _ = model(**batch)
                torch.mps.synchronize()
                prof.step()
                steps += 1
                if steps >= cfg['profile_steps']:
                    break

    rows = []
    for evt in prof.key_averages():
        rows.append({
            'op': evt.key,
            'calls': evt.count,
            'self_cpu_ms': evt.self_cpu_time_total / 1000.0,
            'cpu_total_ms': evt.cpu_time_total / 1000.0,
        })

    rows.sort(key=lambda r: r['self_cpu_ms'], reverse=True)
    return rows[:top_k]

top_ops = profile_bottlenecks(base_model, features, labels, cfg)
top_ops[:5]


[{'op': 'aten::linear',
  'calls': 520,
  'self_cpu_ms': 16.606740000000038,
  'cpu_total_ms': 16.798874000000165},
 {'op': 'aten::copy_',
  'calls': 260,
  'self_cpu_ms': 13.274297999999684,
  'cpu_total_ms': 13.373383999999746},
 {'op': 'aten::_local_scalar_dense',
  'calls': 40,
  'self_cpu_ms': 7.060387999999921,
  'cpu_total_ms': 7.099137000000044},
 {'op': 'aten::_scaled_dot_product_attention_math_for_mps',
  'calls': 80,
  'self_cpu_ms': 5.634416000000022,
  'cpu_total_ms': 5.729207999999957},
 {'op': 'aten::arange',
  'calls': 200,
  'self_cpu_ms': 2.0670950000000907,
  'cpu_total_ms': 4.111658999999991}]

## 4) Сводка результатов
Ниже — итоговые таблицы по базовым метрикам и bottlenecks.

In [7]:
baseline_export = {k: v for k, v in baseline.items() if k not in {'preds', 'logits'}}

baseline_df = pd.DataFrame([baseline_export])
ops_df = pd.DataFrame(top_ops)
ops_df = ops_df[[c for c in ['op', 'calls', 'self_cpu_ms', 'cpu_total_ms'] if c in ops_df.columns]]

print('Baseline metrics')
display(baseline_df.T)

print('Top bottlenecks (profiler)')
display(ops_df)


Baseline metrics


,0
mean_batch_latency_ms,21.296309
p50_batch_latency_ms,21.263354
p95_batch_latency_ms,21.724167
throughput_samples_s,733.695226
accuracy,0.886000
params_million,14.350874
model_size_mb,54.744240
current_memory_mb,55.121338


Top bottlenecks (profiler)


,op,calls,self_cpu_ms,cpu_total_ms
0,aten::linear,520,16.606740,16.798874
1,aten::copy_,260,13.274298,13.373384
2,aten::_local_scalar_dense,40,7.060388,7.099137
3,aten::_scaled_dot_product_attention_math_for_mps,80,5.634416,5.729208
4,aten::arange,200,2.067095,4.111659
5,aten::where,160,1.566104,3.415519
6,aten::fill_,180,1.308310,1.308310
7,aten::index_select,60,1.286717,1.322346
8,aten::gelu,80,1.102010,1.102010
9,aten::all,20,0.793213,0.793213
